# NFCI 表单第二次且最后一次传输修复

前两次分别为 Python requests 读取超时和 curl HTTP/2 流错误，均零响应字节。本版强制 HTTP/1.1，仅同一官方 URL 一次 GET，不请求数据 ZIP 或读取 SOXX 市场结果。

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shlex,subprocess,tarfile,tempfile
from bs4 import BeautifulSoup
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/chicago_nfci_source_preflight_20261023';plan=json.loads((card/'transport_repair2_plan.json').read_text());state=json.loads((card/'repair_run_state.json').read_text())
assert plan['frozen'] and state['status']=='failed_HTTP2_stream_error' and not (card/'repair2_run_state.json').exists()
url=plan['exact_URL'];rawdir=root/'data/snapshots/chicago_nfci_source_preflight_20261023';rawdir.mkdir(parents=True,exist_ok=True)
(card/'repair2_run_state.json').write_text(json.dumps({'status':'direct_curl_GET_started','plan_sha256':hashlib.sha256((card/'transport_repair2_plan.json').read_bytes()).hexdigest(),'additional_GET_attempts':1,'model_fits':0,'accounts':0},indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='nfci-curl-') as tmp:
 header_path=Path(tmp)/'headers.txt';raw_path=Path(tmp)/'form.html';started=datetime.now(timezone.utc).isoformat()
 command=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','120','--fail-with-body','-sS','-L','-D',str(header_path),'-o',str(raw_path),'-H','Accept: text/html','-A','SOXL-Qlib-source-audit/1.0',url]
 process=subprocess.run(command,capture_output=True,text=True,timeout=130);received=datetime.now(timezone.utc).isoformat()
 meta={'url':url,'requested_at_utc':started,'received_at_utc':received,'curl_exit_code':process.returncode,'curl_stderr':process.stderr[:1000],'header_text':header_path.read_text(errors='replace') if header_path.exists() else '', 'bytes':raw_path.stat().st_size if raw_path.exists() else 0}
 (card/'repair2_curl_attempt_receipt.json').write_text(json.dumps(meta,indent=2)+'\n')
 if process.returncode:raise RuntimeError('single frozen direct-curl GET failed; no retry')
 raw=raw_path.read_bytes();assert len(raw)<=plan['budget']['max_response_bytes'] and raw.lstrip().lower().startswith(b'<!doctype html')
 source=rawdir/'alfred_nfci_form.html';source.write_bytes(raw);source_sha=hashlib.sha256(raw).hexdigest()
 soup=BeautifulSoup(raw,'html.parser');assert 'Chicago Fed National Financial Conditions Index' in soup.get_text(' ',strip=True)
 forms=[f for f in soup.find_all('form') if f.find(attrs={'name':'form[file_type]'}) is not None];assert len(forms)==1
 form=forms[0];selector=form.find(attrs={'name':'form[selected_vintage_dates][]'});assert selector
 vintage=[x.get('value') for x in selector.find_all('option') if x.get('value')];assert vintage and len(vintage)==len(set(vintage));vintage=sorted(vintage)
 start=form.find(attrs={'name':'form[obs_start_date]'}).get('value');end=form.find(attrs={'name':'form[obs_end_date]'}).get('value')
 assert vintage[0]<='2011-12-31' and vintage[-1]>='2026-01-01'
 settings=_settings(root);archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:out.add(source,arcname='alfred_nfci_form.html')
 digest=hashlib.sha256(archive.read_bytes()).hexdigest();size=archive.stat().st_size
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 restore=['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes','-o','ConnectTimeout=5',settings['ssh']['host'],'tar -xOzf '+repr(remote)+' alfred_nfci_form.html']
 saved=subprocess.run(restore,capture_output=True,timeout=120);assert saved.returncode==0 and hashlib.sha256(saved.stdout).hexdigest()==source_sha and len(saved.stdout)==len(raw)
 result={'status':'official_ALFRED_NFCI_vintage_form_available_source_only','repair_plan_sha256':hashlib.sha256((card/'transport_repair2_plan.json').read_bytes()).hexdigest(),'form_sha256':source_sha,'form_bytes':len(raw),'observation_start':start,'observation_end':end,'selectable_vintage_dates':len(vintage),'first_vintage_date':vintage[0],'last_vintage_date':vintage[-1],'early_2011_first_public_qualified':False,'nas_archive_sha256':digest,'nas_archive_bytes':size,'nas_artifact':evidence,'nas_remote_hash_verified':True,'scoped_restore_sha256_verified':True,'form_GET_attempts_total':3,'successful_form_GETs':1,'data_ZIP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
 (card/'source_preflight_result.json').write_text(json.dumps(result,indent=2)+'\n')
 (card/'repair2_run_state.json').write_text(json.dumps({'status':'completed','repair_plan_sha256':result['repair_plan_sha256'],'additional_GETs':1,'total_GET_attempts':3,'data_ZIP_GETs':0,'model_fits':0,'accounts':0},indent=2)+'\n')
 print({k:result[k] for k in ('status','selectable_vintage_dates','first_vintage_date','last_vintage_date','observation_start','observation_end')})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


RuntimeError: single frozen direct-curl GET failed; no retry

## 结论边界

若仍失败，记录 needs_source_access，不再重试本来源。若成功，版本列表也只证明历史版本可请求性，不能授予因子或资金账户资格。

最后一次强制 HTTP/1.1 在 120 秒后零字节超时；本来源状态 needs_source_access，零因子/拟合/账户。